In [9]:
import gc
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = Path(Path.cwd()).parent / "Data"
plot_dir = Path(Path.cwd()) / 'Plots'

In [10]:
# Shapefiles
kreise = gpd.read_file(Path(data_dir / "Shapefiles-2022" / "Germany" / "COUNTIES" / "VG250_KRS_clean_final.shp"))

In [11]:
# Translate the R block to pandas
labour_data = pd.read_excel(
    Path(data_dir / "county-wages-2022" / "country-wages-2022-entgelt-dwolk-0-202212-xlsx.xlsx"),
    sheet_name="8.1",
    usecols="A:K",
    skiprows=8,
    nrows=6287,
    dtype=str,
 )
# Stichtag 31.12.2024

# Rename and keep relevant variables (columns 1:4 and 11 in R's 1-based indexing)
labour_data = labour_data.iloc[:, [0, 1, 2, 3, 10]].copy()
labour_data.columns = [
    "AGS",      # Region key (Amtlicher Gemeindeschluessel)
    "region",
    "merkmale",
    "N",        # total workers
    "wages",    # median euro
 ]

# Select counties only
labour_data["AGS"] = labour_data["AGS"].astype("string").str.strip()
labour_data["RS"] = labour_data["AGS"]
labour_data = labour_data[labour_data["AGS"].str.len() == 5].copy()

# Select correct qualifications only
labour_data = labour_data[labour_data["merkmale"].isin([
    "ohne Berufsabschluss",
    "anerkannter Berufsabschluss",
    "akademischer Berufsabschluss",
    "Insgesamt",
])].copy()

labour_data.head()

,AGS,region,merkmale,N,wages,RS
286,01001,"Flensburg, Stadt",Insgesamt,25761,3433.1826196473553,01001
294,01001,"Flensburg, Stadt",ohne Berufsabschluss,2182,2395.6923076923076,01001
295,01001,"Flensburg, Stadt",anerkannter Berufsabschluss,16986,3409.590909090909,01001
296,01001,"Flensburg, Stadt",akademischer Berufsabschluss,4602,4891.728070175439,01001
301,01002,"Kiel, Landeshauptstadt",Insgesamt,77066,3772.814049586777,01002


In [12]:
housing_data = pd.read_csv("county_fixed_effects_demeaned.csv")

In [13]:
housing_data['county_id'] = housing_data['0'].astype(str).str.zfill(5)

In [14]:
data = labour_data
for id in data['AGS'].unique():
    data.loc[data['AGS'] == id, 'price_index'] = housing_data.loc[housing_data['county_id'] == id, 'purchase_price_index'].values[0]


In [ ]:
# If X, then np.nan. If > or < then the number without the symbol.
# Numbers are in format e.g. 2621.298319327731 
def convert_wages(value):
    if value == 'X':
        return np.nan
    elif value.startswith('>'):
        return float(value[1:].strip().replace('.', ''))
    elif value.startswith('<'):
        return float(value[1:].strip().replace('.', ''))
    else:
        return float(value.strip())

v_convert_wages = np.vectorize(convert_wages)
data['wages'] = v_convert_wages(data['wages'])

In [15]:
data

,AGS,region,merkmale,N,wages,RS,price_index
286,01001,"Flensburg, Stadt",Insgesamt,25761,3433.1826196473553,01001,0.922232
294,01001,"Flensburg, Stadt",ohne Berufsabschluss,2182,2395.6923076923076,01001,0.922232
295,01001,"Flensburg, Stadt",anerkannter Berufsabschluss,16986,3409.590909090909,01001,0.922232
296,01001,"Flensburg, Stadt",akademischer Berufsabschluss,4602,4891.728070175439,01001,0.922232
301,01002,"Kiel, Landeshauptstadt",Insgesamt,77066,3772.814049586777,01002,1.094564
...,...,...,...,...,...,...,...
6266,16076,Greiz,akademischer Berufsabschluss,2042,4233,16076,0.003731
6271,16077,Altenburger Land,Insgesamt,17598,2745.2963800904977,16077,0.172768
6279,16077,Altenburger Land,ohne Berufsabschluss,881,2320.1428571428573,16077,0.172768
6280,16077,Altenburger Land,anerkannter Berufsabschluss,13794,2703.4262086513995,16077,0.172768


In [ ]:
data['qol'] = (-1) * np.log(data['wages']) + 1/3 * data['price_index']

TypeError: loop of ufunc does not support argument 0 of type str which has no callable log method